# INAIT FutureComplete API — Integration Example

This notebook demonstrates how to use the INAIT FutureComplete forecasting API
to submit time series data, run predictions and backtests, and retrieve results.

This is the same workflow the **Inait-FutureComplete Teams Agent** uses behind the scenes.

## API Workflow

```
1. POST /v1/prediction (or /v1/backtest)  → session_id
2. GET  /v1/sessions/{session_id}/status  → poll until 'completed'
3. GET  /v1/sessions/{session_id}/result  → fetch results
```

In [ ]:
import requests
import time
import pandas as pd
from typing import Optional

## Configuration

In [ ]:
# INAIT API credentials
BASE_URL = "https://api.forecasting.inait.ai"
API_KEY = "7e7215551bbc497c807868759bfa3c16"

API_VERSION = "v1"
API_ENDPOINTS = {
    "prediction": f"/{API_VERSION}/prediction",
    "backtest": f"/{API_VERSION}/backtest",
    "benchmark": f"/{API_VERSION}/benchmark",
    "session_status": f"/{API_VERSION}/sessions/{{session_id}}/status",
    "session_result": f"/{API_VERSION}/sessions/{{session_id}}/result",
    "artifacts": f"/{API_VERSION}/artifacts/{{resource_id}}",
}

HEADERS = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": API_KEY,
}

## Helper Functions

In [ ]:
def submit_job(endpoint_key: str, payload: dict) -> str:
    """Submit a job to INAIT API and return session_id."""
    url = f"{BASE_URL}{API_ENDPOINTS[endpoint_key]}"
    response = requests.post(url, json=payload, headers=HEADERS, timeout=60)
    response.raise_for_status()
    data = response.json()
    return data["response"]["session_id"]


def get_status(session_id: str) -> str:
    """Check job status."""
    url = f"{BASE_URL}{API_ENDPOINTS['session_status'].format(session_id=session_id)}"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()["status"]


def get_result(session_id: str) -> dict:
    """Fetch completed job results."""
    url = f"{BASE_URL}{API_ENDPOINTS['session_result'].format(session_id=session_id)}"
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    return response.json()


def poll_until_complete(session_id: str, max_attempts=60, interval=30) -> dict:
    """Poll until job completes, then return results."""
    for attempt in range(max_attempts):
        status = get_status(session_id)
        print(f"  [{attempt+1}/{max_attempts}] Status: {status}")
        if status == "completed":
            return get_result(session_id)
        elif status == "failed":
            raise Exception(f"Job {session_id} failed!")
        time.sleep(interval)
    raise TimeoutError(f"Job did not complete in {max_attempts * interval}s")

## Load Sample Data

We use the AAPL/MSFT volatility dataset from the INAIT examples.

In [ ]:
# Load data from the INAIT public examples repo
data_url = "https://raw.githubusercontent.com/inait-external/inait-forecast-docs/main/data/dataset_GKYZ_2016_AAPL_MSFT_trimmed.csv"
data = pd.read_csv(data_url, index_col=0)
print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
print(f"\nColumns ({len(data.columns)}):")
for col in data.columns:
    print(f"  - {col}")
data.head()

## Define Forecasting Configuration

For this example:
- **Targets:** `AAPL` (Apple volatility)
- **Features:** All `AAPL__*` columns (local features for AAPL)
- **Horizon:** 3 days ahead

In [ ]:
# Define targets and features
targets = "AAPL"

# Use AAPL-specific local features
aapl_features = [
    "AAPL__hl", "AAPL__co", "AAPL__ocp",
    "AAPL__log_returns", "AAPL__lret_squared",
    "AAPL__earning_horizon1", "AAPL__earning_horizon2",
    "AAPL__earning_horizon3", "AAPL__earning_horizon4",
]
features = ",".join(aapl_features)

forecasting_horizon = 3

print(f"Target: {targets}")
print(f"Features: {features}")
print(f"Horizon: {forecasting_horizon} days")

## Submit a Forecast Job

In [ ]:
# Build the forecast payload
payload_forecast = {
    "data": data.to_dict(orient="split"),
    "config": {
        "operation": "forecast",
        "operation_arguments": {
            "operation_type": "forecast",
            "forecasting_horizon": forecasting_horizon,
            "targets": targets,
            "features": features,
            "prediction_interval_levels": None,
            "prediction_stride": 1,
            "end_date": None,
            "run_explain": False,
            "include_genai_summary": False,
        },
    },
    "background": True,
}

print("Submitting forecast job...")
session_id = submit_job("prediction", payload_forecast)
print(f"Job submitted! Session ID: {session_id}")

In [ ]:
# Poll for results
print("Waiting for forecast to complete...")
result = poll_until_complete(session_id)
print("\nForecast complete!")

In [ ]:
# Parse and display prediction results
predictions = pd.DataFrame(**result["response"]["data"]["predictions"])
print(f"Predictions shape: {predictions.shape}")
predictions

## Submit a Backtest Job

Backtest evaluates the model on a held-out historical period.

In [ ]:
# Build backtest payload
payload_backtest = {
    "data": data.to_dict(orient="split"),
    "config": {
        "operation": "backtest",
        "operation_arguments": {
            "operation_type": "backtest",
            "forecasting_horizon": forecasting_horizon,
            "targets": targets,
            "features": features,
            "prediction_interval_levels": "50,90",
            "prediction_stride": 1,
            "run_explain": False,
            "include_genai_summary": False,
            "backtest_size": None,
            "start_date": "2016-11-01",
            "end_date": "2016-12-31",
        },
    },
    "background": True,
}

print("Submitting backtest job...")
session_id_bkt = submit_job("backtest", payload_backtest)
print(f"Job submitted! Session ID: {session_id_bkt}")

In [ ]:
# Poll for backtest results
print("Waiting for backtest to complete...")
result_bkt = poll_until_complete(session_id_bkt)
print("\nBacktest complete!")

In [ ]:
# Parse backtest results
preds_bkt = result_bkt["response"]["data"]["predictions"]
scores_bkt = result_bkt["response"]["data"]["scores"]

pred_df = pd.DataFrame(**preds_bkt[0])
print(f"Backtest predictions shape: {pred_df.shape}")
print(f"\nScores:")
scores_df = pd.DataFrame(**scores_bkt)
scores_df

## Cross Learning Example (Multi-Target)

Forecast both AAPL and MSFT jointly for improved accuracy.

In [ ]:
# Cross Learning: all columns as features for both targets
features_cl = ",".join(data.columns.difference(["AAPL", "MSFT"]))

payload_cl = {
    "data": data.to_dict(orient="split"),
    "config": {
        "operation": "forecast",
        "operation_arguments": {
            "operation_type": "forecast",
            "forecasting_horizon": 10,
            "targets": "AAPL, MSFT",
            "features": features_cl,
            "prediction_interval_levels": None,
            "prediction_stride": 1,
            "end_date": None,
            "run_explain": True,
            "include_genai_summary": False,
        },
    },
    "background": True,
}

print("Submitting Cross Learning forecast (AAPL + MSFT)...")
session_id_cl = submit_job("prediction", payload_cl)
print(f"Job submitted! Session ID: {session_id_cl}")

In [ ]:
# Poll for CL results
print("Waiting for Cross Learning forecast to complete...")
result_cl = poll_until_complete(session_id_cl)
print("\nCross Learning forecast complete!")

# Results may be offloaded for large responses
resource_id = result_cl.get("response", {}).get("resource_id")
if resource_id:
    print(f"Results offloaded. Resource ID: {resource_id}")
    # Download offloaded results
    url = f"{BASE_URL}{API_ENDPOINTS['artifacts'].format(resource_id=resource_id)}"
    resp = requests.get(url, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    cl_data = resp.json()
    pred_df_cl = pd.DataFrame(**cl_data["predictions"])
    print(f"CL Predictions shape: {pred_df_cl.shape}")
    pred_df_cl.head()
else:
    pred_df_cl = pd.DataFrame(**result_cl["response"]["data"]["predictions"])
    print(f"CL Predictions shape: {pred_df_cl.shape}")
    pred_df_cl.head()

## Summary

This notebook demonstrated:
1. **Loading data** in the required wide tabular format
2. **Submitting forecast jobs** via `POST /v1/prediction`
3. **Polling for completion** via `GET /v1/sessions/{id}/status`
4. **Retrieving results** via `GET /v1/sessions/{id}/result`
5. **Backtesting** with prediction intervals
6. **Cross Learning** with multiple targets

The Teams agent (`src/agent.py`) wraps this same logic in a conversational interface,
allowing CFOs to upload data, configure parameters, and receive results as Teams messages.

**Portal:** [https://futurecomplete.inait.ai](https://futurecomplete.inait.ai)